# preEHA — Preparar datos para Event History Analysis

Este notebook hace únicamente **preparación de datos**.

Su producto final es una base en la que cada gabinete tiene:

- una duración observada;
- un indicador de si la caída fue observada o si el caso quedó censurado;
- las variables políticas que utilizaremos después.

El análisis de supervivencia se realiza en el notebook **EHA**.

## preEHA-01 — Leer los datos sin procesar

In [ ]:
import pandas as pd

# Sustituir por la URL del archivo sin procesar cuando se publique.
url_raw = "PEGAR_AQUI_URL_PICKLE_SIN_PROCESAR"

gabinetes = pd.read_pickle(url_raw)
gabinetes.head()

## preEHA-02 — Revisar fechas y situación final

In [ ]:
gabinetes[
    ["fecha_inicio", "fecha_fin_observacion", "situacion_final"]
].head()

La fecha final no significa siempre que el gabinete haya caído.

Si `situacion_final` es `vigente_al_cierre`, solo sabemos que el gabinete sobrevivió hasta esa fecha.
Ese caso será **censurado a la derecha**.

## preEHA-03 — Construir duración y evento

In [ ]:
gabinetes["duracion_meses"] = (
    (gabinetes["fecha_fin_observacion"] - gabinetes["fecha_inicio"]).dt.days
    / 30.4375
).round(1)

gabinetes["caida_evento"] = (
    gabinetes["situacion_final"] == "caida"
).astype(int)

gabinetes[
    ["duracion_meses", "caida_evento"]
].head()

## preEHA-04 — Codificar gobierno mayoritario

In [ ]:
gabinetes["gobierno_mayoritario"] = (
    gabinetes["tipo_gobierno"] == "mayoritario"
).astype(int)

gabinetes[
    ["tipo_gobierno", "gobierno_mayoritario"]
].head()

## preEHA-05 — Guardar la base lista para EHA

In [ ]:
eha = gabinetes[
    [
        "id_gabinete",
        "duracion_meses",
        "caida_evento",
        "gobierno_mayoritario",
        "fragmentacion",
        "crecimiento_pbi"
    ]
].copy()

eha.to_pickle("eha_procesada.pkl")

eha.head()

## Producto de preEHA

La base `eha_procesada.pkl` está lista para análisis de supervivencia.

El notebook **EHA** no volverá a calcular duración, censura ni codificaciones.